In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybop
import pybamm

In [ ]:
# Data is up two directory then in data/pybamm/
data_path = Path.cwd().parent / "data" / "pybamm"/"LGM50_789_1C_25degC.csv"
data = pd.read_csv(data_path)

# This data has some issues in the time column, so we will fix it
mask = data["Time [s]"].values[:-1] < data["Time [s]"].values[1:]  # Check if the time is increasing

# where the mask is false, we will drop the row
data = data.iloc[:-1][mask]


# Transform the data into a pybop dataset
dataset = pybop.Dataset(
    {
     "Time [s]": data["Time [s]"].values,
     "Voltage [V]": data["Voltage [V]"].values,
     "X-averaged cell temperature [K]": data["X-averaged cell temperature [K]"].values,
     "Current function [A]": np.ones_like(data["Voltage [V]"].values)
    }
)


In [ ]:
model = pybop.lithium_ion.SPMe(
    name="TSPMe",
    options={
        "thermal": "lumped",
        "dimensionality": 0,
        "cell geometry": "arbitrary",
        "electrolyte conductivity": "integrated",
        }
)

In [ ]:
# Define the model and parameter set
parameter_set = pybop.ParameterSet.pybamm("Chen2020")
# Define the operating conditions
experiment = pybamm.Experiment(
    [
        "Discharge at 1C until 2.5 V",
        "Rest for 2 hours",
    ],
    period="30 seconds",
)
